# 02 — Model report (M3–M4)

Baseline (logistic regression) vs the **naive scout** (rank by market value at cutoff age) vs
**CatBoost**. Temporal split by birth-year cohort. Metrics: PR-AUC, ROC-AUC, Recall@Top-K,
Brier, calibration curve; SHAP importance for CatBoost.

Loads `data/processed/features.parquet` (full `run_ingest`) or the demo sample. **With the demo
sample the numbers are degenerate** (almost no negatives) — wiring check only; the real
comparison needs the full dataset. Run `scripts/run_gbm.py` first to populate MLflow.

In [ ]:
import sys

sys.path.insert(0, "..")
from pathlib import Path

import pandas as pd

from eval.metrics import calibration_table, evaluate_binary
from features.split import temporal_split
from models.baseline import LogRegBaseline, naive_scout_scores
from settings import load_settings

proc = Path("../data/processed")
path = proc / "features.parquet"
if not path.exists():
    path = proc / "features_demo.parquet"
df = pd.read_parquet(path)
test_from = load_settings()["split"]["test_cohort_from"]
train, test = temporal_split(df, test_cohort_from=test_from)
print(
    path.name,
    "| train",
    len(train),
    "test",
    len(test),
    "| test base rate",
    f"{test['target'].mean():.1%}",
)

## M3 — baseline vs naive scout

In [ ]:
model = LogRegBaseline().fit(train, train["target"])
p_model = model.predict_proba(test)
s_scout = naive_scout_scores(test)

k = min(20, len(test))
rows = {
    "logreg baseline": evaluate_binary(test["target"], p_model, k=k),
    "naive scout": evaluate_binary(test["target"], s_scout, k=k),
}
pd.DataFrame(rows).T.reindex(columns=["pr_auc", "roc_auc", f"recall_at_{k}", "brier"])

In [ ]:
tbl = calibration_table(test["target"], p_model, n_bins=8)
if tbl:
    import matplotlib.pyplot as plt

    mp = [r[3] for r in tbl]
    fp = [r[4] for r in tbl]
    fig, ax = plt.subplots(figsize=(4, 4))
    ax.plot([0, 1], [0, 1], "--", c="grey")
    ax.plot(mp, fp, "o-")
    ax.set_xlabel("mean predicted")
    ax.set_ylabel("observed frac positive")
    ax.set_title("calibration — logreg baseline")
    fig.tight_layout()
else:
    print("not enough data for a calibration curve")

## M4 — CatBoost + SHAP

In [ ]:
from features.build_features import feature_columns
from models.gbm import CatBoostBreakthrough
from eval.shap_analysis import mean_abs_importance

feats = feature_columns(df)
gbm = CatBoostBreakthrough(params={"iterations": 200}).fit(train[feats], train["target"])
res = {
    "catboost": evaluate_binary(test["target"], gbm.predict_proba(test[feats]), k=k),
    "logreg": evaluate_binary(test["target"], p_model, k=k),
    "naive scout": evaluate_binary(test["target"], s_scout, k=k),
}
display(pd.DataFrame(res).T.reindex(columns=["pr_auc", "roc_auc", f"recall_at_{k}", "brier"]))

imp = mean_abs_importance(gbm, train[feats])
imp.head(15).iloc[::-1].plot(kind="barh", figsize=(6, 5), title="mean |SHAP| — CatBoost")

In [ ]:
import mlflow

try:
    mlflow.set_tracking_uri("sqlite:///../mlruns/mlflow.db")
    runs = mlflow.search_runs(experiment_names=["rpl-breakthrough"])
    metric_cols = [c for c in runs.columns if c.startswith("metrics.")]
    display(runs[["run_id", *metric_cols]] if len(runs) else "no runs yet — run scripts/run_gbm.py")
except Exception as e:
    print("mlflow:", e)